# 🎬 CineScope: Core Data, Optimization & Model Training Pipeline
This notebook serves as the interactive testing ground, hyperparameter tuning workspace, and offline pre-computation pipeline for the CineScope Hybrid Recommendation Engine. It executes empirical baseline comparisons, searches parameter spaces using Grid Search Cross-Validation, extracts TF-IDF metadata matrices, and serializes production-ready model artifacts.

In [ ]:
import sys
import os
from pathlib import Path
from collections import defaultdict
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly, SVD, accuracy
from surprise.model_selection import train_test_split, GridSearchCV

# Configure plotting aesthetics
sns.set_theme(style="whitegrid", palette="muted")

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

In [ ]:
# Universal project anchors dynamically resolved
ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data" / "ml-latest-small"
MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_FILE = DATA_DIR / 'ratings.csv'
MOVIES_FILE = DATA_DIR / 'movies.csv'

if not RATINGS_FILE.exists() or not MOVIES_FILE.exists():
    raise FileNotFoundError("Dataset targets missing from disk. Ensure data path alignment is correct.")

print(f"Root Project Workspace: {ROOT_DIR}")
print(f"Data Directory Integrity Verified: {DATA_DIR.exists()}")

### 📊 Exploratory Data Analysis (EDA): Rating Behaviors

In [ ]:
print("📊 Rendering Rating Distribution Matrix...")
ratings_df = pd.read_csv(RATINGS_FILE)

plt.figure(figsize=(10, 5))
ax = sns.countplot(x='rating', data=ratings_df, palette='viridis')
plt.title('Distribution of Movie Ratings in Target Dataset', pad=15, fontweight='bold')
plt.xlabel('Rating (0.5 to 5.0)', fontweight='bold')
plt.ylabel('Interaction Count', fontweight='bold')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=9, xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

### 📈 Interaction Sparsity & The Long Tail
Collaborative filtering engines rely on dense interaction matrices. This plot proves the existence of the "Long Tail"—where a tiny fraction of blockbusters hold the majority of ratings, while thousands of obscure movies have almost no data. This justifies our need for advanced Matrix Factorization (SVD) to infer latent factors for sparse items.

In [ ]:
print("📉 Analyzing Interaction Sparsity (The Long Tail)...")
movie_popularity = ratings_df.groupby('movieId').size().sort_values(ascending=False).values

plt.figure(figsize=(10, 5))
plt.plot(movie_popularity, color='crimson', linewidth=2)
plt.fill_between(range(len(movie_popularity)), movie_popularity, color='crimson', alpha=0.3)
plt.title('The Long Tail of Movie Interactions', pad=15, fontweight='bold')
plt.xlabel('Movie Rank (Most to Least Rated)', fontweight='bold')
plt.ylabel('Total Number of Ratings', fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

total_possible = ratings_df['userId'].nunique() * ratings_df['movieId'].nunique()
sparsity = (1 - (len(ratings_df) / total_possible)) * 100
print(f"Matrix Sparsity Calculated: {sparsity:.2f}% of the user-item matrix is entirely empty.")

### 🎭 Genre Distribution Analysis
To build our Content-Based pipeline, we must understand the categorical distribution of our metadata. High-frequency genres (like Drama and Comedy) offer little personalization value. This imbalance justifies the use of TF-IDF, which will mathematically penalize ubiquitous genres and boost the weight of niche categories (like Film-Noir or IMAX).

In [ ]:
print("🎭 Analyzing Genre Frequencies...")
movies_df = pd.read_csv(MOVIES_FILE)

all_genres = movies_df['genres'].str.split('|').explode()
all_genres = all_genres[all_genres != '(no genres listed)']
genre_counts = all_genres.value_counts()

plt.figure(figsize=(12, 6))
ax = sns.barplot(y=genre_counts.index, x=genre_counts.values, palette='magma')
plt.title('Categorical Genre Distribution in MovieLens Dataset', pad=15, fontweight='bold')
plt.xlabel('Number of Movies', fontweight='bold')
plt.ylabel('Genre', fontweight='bold')

for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', fontsize=9, xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
plt.show()

### 📉 Empirical Predictive Error Baseline Comparisons

In [ ]:
print("📊 Loading MovieLens rating stream for baseline assessment...")
reader = Reader(rating_scale=(0.5, 5.0))
baseline_data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)

print("✂️ Partitioning data into unified 80% train / 20% test splits...")
trainset_b, testset_b = train_test_split(baseline_data, test_size=0.2, random_state=42)
baseline_results = {}

algo_random = NormalPredictor()
algo_random.fit(trainset_b)
baseline_results['Random Guess'] = accuracy.rmse(algo_random.test(testset_b), verbose=False)

bsl_options = {'method': 'als', 'n_epochs': 5, 'reg_u': 12, 'reg_i': 5}
algo_baseline = BaselineOnly(bsl_options=bsl_options)
algo_baseline.fit(trainset_b)
baseline_results['Statistical Average'] = accuracy.rmse(algo_baseline.test(testset_b), verbose=False)

algo_vanilla_svd = SVD(random_state=42)
algo_vanilla_svd.fit(trainset_b)
baseline_results['Untuned SVD'] = accuracy.rmse(algo_vanilla_svd.test(testset_b), verbose=False)

print("\n🏆 INITIAL RMSE PARADIGM PERFORMANCE COMPARISON")
for model, rmse_val in sorted(baseline_results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model:<25} : {rmse_val:.4f}")

models = list(baseline_results.keys())
rmse_scores = list(baseline_results.values())

plt.figure(figsize=(10, 4))
ax = sns.barplot(x=rmse_scores, y=models, hue=models, palette='rocket', legend=False)
plt.title('Baseline RMSE Performance (Lower is Better)', pad=15, fontweight='bold')
plt.xlabel('Root Mean Square Error (RMSE)', fontweight='bold')
plt.xlim(0.8, 1.5)

for i, v in enumerate(rmse_scores):
    ax.text(v + 0.01, i, f"{v:.4f}", color='black', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### ⚙️ Hyperparameter Optimization and Grid Search

In [ ]:
print("⚙️ Executing Grid Search Cross-Validation over defined hyperparameter partitions...")
param_grid = {
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.05, 0.06, 0.1]
}

grid_search = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
grid_search.fit(baseline_data)

print(f"🏆 Target Tuning Complete. Absolute Best Achieved RMSE: {grid_search.best_score['rmse']:.4f}")
optimal_params = grid_search.best_params['rmse']
print(f"💡 Mathematically Optimal Structural Coordinates Found:\n{optimal_params}")

### 🧠 Build Content-Based Filtering Matrices (TF-IDF)

In [ ]:
print("Parsing movies dataset metadata...")
movies_df['genres_space'] = movies_df['genres'].str.replace('|', ' ', regex=False).fillna('')

print("Vectorizing genre matrices with TF-IDF...")
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(movies_df['genres_space'])

print(f"Matrix construction finalized. Structural Shape: {genre_matrix.shape}")

### 🤖 Train Optimized Collaborative Filtering Parameters

In [ ]:
print("Instantiating final SVD engine using optimized parameter maps...")
svd_engine = SVD(
    n_epochs=optimal_params['n_epochs'],
    lr_all=optimal_params['lr_all'],
    reg_all=optimal_params['reg_all'],
    random_state=42
)

print("Factoring explicit feedback matrices over complete training volume...")
full_trainset = baseline_data.build_full_trainset()
svd_engine.fit(full_trainset)

print("SVD parameter optimization complete.")

### 📊 Run Evaluation Metrics Suite (Precision, Recall, NDCG)

In [ ]:
eval_model = SVD(
    n_epochs=optimal_params['n_epochs'],
    lr_all=optimal_params['lr_all'],
    reg_all=optimal_params['reg_all'],
    random_state=42
)
eval_model.fit(trainset_b)
test_predictions = eval_model.test(testset_b)

def eval_pipeline(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    ndcgs = []
    
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
        
        if len(user_ratings) >= 2:
            dcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            user_ratings.sort(key=lambda x: x[1], reverse=True)
            idcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            if idcg > 0:
                ndcgs.append(dcg / idcg)

    return np.mean(list(precisions.values())), np.mean(list(recalls.values())), np.mean(ndcgs)

mean_precision, mean_recall, mean_ndcg = eval_pipeline(test_predictions)

print(f"• Mean Precision@10: {mean_precision:.4f}")
print(f"• Mean Recall@10:    {mean_recall:.4f}")
print(f"• Mean NDCG@10:      {mean_ndcg:.4f}")

metrics = ['Precision@10', 'Recall@10', 'NDCG@10']
scores = [mean_precision, mean_recall, mean_ndcg]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=metrics, y=scores, hue=metrics, palette='mako', legend=False)
plt.title('Top-10 Recommendation Engine Performance Metrics', pad=15, fontweight='bold')
plt.ylabel('Score (0.0 to 1.0)', fontweight='bold')
plt.ylim(0, 1.0)

for i, v in enumerate(scores):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

### 💾 Serialize Final Engine Artifacts

In [ ]:
print("Exporting structural data binaries to target storage storage locations...")

with open(MODEL_DIR / 'movies.pkl', 'wb') as f:
    pickle.dump(movies_df, f)

with open(MODEL_DIR / 'genre_matrix.pkl', 'wb') as f:
    pickle.dump(genre_matrix, f)

with open(MODEL_DIR / 'svd_model.pkl', 'wb') as f:
    pickle.dump(svd_engine, f)

print("All binary engine artifacts saved successfully. Ready for Streamlit UI deployment.")